# 42 — Clustering: k-means & GMM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/042-clustering-kmeans-gmm.ipynb)

k-means alternates two moves: assign each point to its nearest centroid, then move each centroid to the mean of its points. In this notebook you run those moves on six points, watch the inertia fall, and cluster two blobs in 2-D.

In [ ]:
import numpy as np                                        # the array library
X = np.array([[1.], [2.], [3.], [10.], [11.], [12.]])     # six points, one feature
C = np.array([[1.], [12.]])                               # two starting centroids
for step in range(3):                                     # three passes at most
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)    # squared distances, 6x2
    a = d.argmin(1)                                       # nearest centroid per point
    C = np.array([X[a == k].mean(0) for k in range(2)])   # recentre each cluster
    inertia = ((X - C[a]) ** 2).sum()                     # within-cluster sum of squares
    print(step, a, C.ravel(), round(float(inertia), 3))   # labels, centres, inertia


## What the numbers say

After the first pass the labels are `[0 0 0 1 1 1]`, the centres are `[2, 11]`, and the inertia is 4.0. The second pass changes nothing, so the loop has converged. Because the update uses the mean, a centroid can only sit at the middle of its own points.

In [ ]:
import matplotlib.pyplot as plt                           # plotting library
rng = np.random.default_rng(0)                            # a fixed random seed
A = rng.normal([0, 0], 0.6, size=(40, 2))                 # first blob of 40 points
B = rng.normal([4, 4], 0.6, size=(40, 2))                 # second blob of 40 points
X2 = np.vstack([A, B])                                    # stack them into 80 rows
C2 = X2[[0, 50]]                                          # two seeds, one per blob
for _ in range(6):                                        # six Lloyd iterations
    d2 = ((X2[:, None, :] - C2[None, :, :]) ** 2).sum(-1) # pairwise squared distances
    a2 = d2.argmin(1)                                     # assign to the nearest centre
    C2 = np.array([X2[a2 == k].mean(0) for k in range(2)])  # move centres to the means
plt.scatter(X2[:, 0], X2[:, 1], c=a2, cmap="coolwarm", s=14)  # colour by cluster
plt.scatter(C2[:, 0], C2[:, 1], marker="x", s=150, color="black")  # draw centres
plt.title("k-means on two round blobs")                    # the takeaway
plt.show()                                                # render the figure


## Your turn

The blobs are round and similarly sized, which is exactly k-means' assumption. Stretch one blob by multiplying its second coordinate by 5, rerun, and watch the boundary cut across the long axis instead of between the blobs. That failure is the reason a Gaussian mixture with full covariance exists.

In [ ]:
k = 2                                                     # number of clusters
inertias = []                                             # collect one value per k
for k in range(1, 6):                                     # try k from 1 to 5
    Ck = X2[np.linspace(0, len(X2) - 1, k).astype(int)]    # spread the seeds out
    for _ in range(6):                                    # a few Lloyd passes
        dk = ((X2[:, None, :] - Ck[None, :, :]) ** 2).sum(-1)  # distances to centres
        ak = dk.argmin(1)                                 # nearest-centre labels
        Ck = np.array([X2[ak == j].mean(0) for j in range(k)])  # update centres
    inertias.append(((X2 - Ck[ak]) ** 2).sum())           # inertia for this k
plt.plot(range(1, 6), inertias, marker="o")               # elbow-style curve
plt.xlabel("k")                                          # x-axis label
plt.ylabel("inertia")                                     # y-axis label
plt.title("Inertia always falls as k rises")              # the takeaway
plt.show()                                                # render the figure
